# J16 — Vérifier les données et expliquer simplement les résultats

## Objectifs

- Vérifier que les fichiers J15a et J15b correspondent aux manifestes.
- Contrôler les communes, les IRIS et les rattachements géographiques.
- Repérer les libellés, unités et sources à compléter.
- Vérifier les valeurs numériques et quelques calculs compréhensibles.
- Préparer les tests manuels de l’application.
- Produire un bilan en français, facile à reprendre dans un compte rendu.

## Périmètre

Les exports locaux utilisés par l’application, sur l’ensemble des territoires qu’ils contiennent.
Cergy et Pontoise servent de premiers exemples de lecture.

## Interprétation

**OK** signifie que la règle testée est respectée.
**À vérifier** demande une relecture ou une explication.
**À corriger** signale une incohérence à traiter.
**Non testé** indique un contrôle non réalisé ou une vérification humaine nécessaire.

Le bilan ne modifie ni les valeurs, ni les unités, ni les statuts de validation du catalogue.
Les étapes commerces/cinémas, GTFS, fréquentation et zones accessibles restent prévues aux J17 à J21.

## 1 — Préparer les imports

Cette cellule corrige les variables de threads avant d’importer NumPy et pandas,
comme dans les précédents notebooks de ton environnement Anaconda.

In [3]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

from pathlib import Path
import importlib.util
import json
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(objet):
        print(objet)

pd.set_option("display.max_columns", 10)
pd.set_option("display.max_rows", 30)
pd.set_option("display.max_colwidth", 110)
print("Imports chargés ✅")

Imports chargés ✅


## 2 — Indiquer la racine et les dossiers d’exports

Extraire le ZIP J16 à la racine de `GeoMarketing_IDF` : le fichier
`scripts/controle_j16.py` doit être présent.

Renseigner les chemins manuels uniquement si les valeurs détectées ne sont pas celles à contrôler.
Le dossier habituel est `data/app/j15b`. L’organisation de ton dépôt déployé peut être
`app/data/app/j15b`. Le contrôle affiche le dossier effectivement choisi.

In [5]:
RACINE_MANUELLE = ""  # Exemple : r"C:/Users/almou/OneDrive/GeoMarketing_IDF"
DOSSIER_J15A_MANUEL = ""  # Exemple : "data/app/j15" ou "app/data/app/j15"
DOSSIER_J15B_MANUEL = ""  # Exemple : "data/app/j15b" ou "app/data/app/j15b"
FOCUS = ["95127", "95500"]  # Cergy, Pontoise

candidats = ([Path(RACINE_MANUELLE)] if RACINE_MANUELLE else
    ([Path(os.environ["GEOMARKETING_RACINE"])] if os.environ.get("GEOMARKETING_RACINE") else
     [Path.cwd(), *Path.cwd().parents, Path(r"C:/Users/almou/OneDrive/GeoMarketing_IDF")]))
RACINE = next((p.resolve() for p in candidats if (p/"scripts/controle_j16.py").is_file()), None)
if RACINE is None:
    raise FileNotFoundError("Extraire le ZIP J16 à la racine du projet puis renseigner RACINE_MANUELLE.")

spec = importlib.util.spec_from_file_location("controle_j16_local", RACINE/"scripts/controle_j16.py")
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
DOSSIER_J15A, DOSSIER_J15B = module.choisir_dossiers(
    RACINE, DOSSIER_J15A_MANUEL, DOSSIER_J15B_MANUEL)
audit = module.ControleJ16(RACINE, DOSSIER_J15A, DOSSIER_J15B, focus=FOCUS)

print("Racine :", RACINE)
print("J15a contrôlé :", DOSSIER_J15A)
print("J15b contrôlé :", DOSSIER_J15B)
print("Tableaux de contrôle :", audit.interim)
print("Compte rendu :", audit.rapports)

Racine : C:\Users\almou\OneDrive\GeoMarketing_IDF
J15a contrôlé : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\app\j15
J15b contrôlé : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\app\j15b
Tableaux de contrôle : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\interim\j16
Compte rendu : C:\Users\almou\OneDrive\GeoMarketing_IDF\outputs\rapports\j16


## 3 — Vérifier que les fichiers vont bien ensemble

**Ce que l’on regarde :** les fichiers sont-ils présents et identiques à ceux décrits par les manifestes ?

**Pourquoi :** un catalogue modifié sans son manifeste peut empêcher Streamlit de charger les données.

**Comment lire :** J15a utilise le schéma 2 et J15b le schéma 1. Cette différence est normale.
Une erreur d’empreinte se traite en restaurant ou régénérant le lot cohérent.

In [6]:
display(audit.verifier_fichiers())

,Contrôle,Résultat,Nombre concerné,Ce que cela signifie,Action à faire
0,J15A : manifeste,OK,0,Le manifeste décrit 12 fichiers. Les versions J15a et J15b sont volontairement différentes.,
1,J15A : fichiers du lot,OK,0,Chaque fichier correspond exactement à la version décrite par son manifeste.,
2,J15B : manifeste,OK,0,Le manifeste décrit 25 fichiers. Les versions J15a et J15b sont volontairement différentes.,
3,J15B : fichiers du lot,OK,0,Chaque fichier correspond exactement à la version décrite par son manifeste.,
4,Synchronisation J15a et J15b,OK,0,Les indicateurs détaillés utilisent bien la base cartographique choisie.,


## 4 — Vérifier les communes et les IRIS

**Ce que l’on regarde :** codes uniques, formats, nombre de territoires et rattachements.

**Pourquoi :** une mauvaise jointure peut doubler une commune ou affecter un indicateur au mauvais territoire.

Les codes restent du texte. Le nombre de communes dépend de la géographie du projet :
il n’est pas fixé arbitrairement à 1 266 ou 1 267.
Le rattachement d’un IRIS utilise le code de contexte du projet, utile pour Paris et les fusions communales.

In [7]:
display(audit.verifier_territoires())

,Contrôle,Résultat,Nombre concerné,Ce que cela signifie,Action à faire
5,profil_communes : codes,OK,0,"1267 lignes ; 0 codes invalides, hors IDF ou répétés.",
6,profil_communes : nombre de territoires,OK,0,1267 lignes lues ; 1267 annoncées par le manifeste.,
7,profil_iris : codes,OK,0,"5264 lignes ; 0 codes invalides, hors IDF ou répétés.",
8,profil_iris : nombre de territoires,OK,0,5264 lignes lues ; 5264 annoncées par le manifeste.,
9,Rattachement des IRIS aux communes,OK,0,0 IRIS rattaché(s) à une commune absente du profil.,
10,Communes de démonstration,OK,0,Cergy et Pontoise sont présentes dans le profil.,


## 5 — Vérifier les libellés, unités et définitions

**Ce que l’on regarde :** un lecteur comprend-il ce qui est mesuré ?

| Indicateur | Exemple d’unité attendue |
| :--- | :--- |
| Population | Habitants ou personnes, selon la source |
| Part des 15-24 ans | % |
| Nombre de ménages | Ménages |
| Densité de restaurants | Restaurants pour 10 000 habitants |
| Écart entre deux pourcentages | Points de pourcentage |

Un contrôle signale les champs manquants et les unités encore « à préciser ».
Il ne choisit pas une unité à la place de la source.
Les codes comme `J13B_...` restent disponibles dans `COLONNE` ; ils doivent disparaître du libellé affiché.

In [8]:
display(audit.verifier_catalogue())
documentation = pd.DataFrame(audit.documentation)
if not documentation.empty:
    a_revoir = documentation.loc[
        documentation["CHAMPS_A_COMPLETER"].ne("") |
        documentation["LIBELLE_A_RELIRE"].ne("")]
    print("Premières variables à documenter ou à relire :")
    display(a_revoir.head(20))

,Contrôle,Résultat,Nombre concerné,Ce que cela signifie,Action à faire
11,Identifiants et structure,OK,0,1348 variables ; 0 ligne(s) avec identifiant répété ou valeur de structure invalide.,
12,Nombre de variables,OK,0,1348 variables dans le catalogue ; 1348 dans le manifeste.,
13,Libellés compréhensibles,À vérifier,104,104 libellé(s) à relire selon les règles de présentation.,Consulter documentation_indicateurs_j16.csv ; appliquer le correctif J15b puis relire les libellés concernés.
14,"Sources, années, définitions et unités",À vérifier,1348,1348 variable(s) avec documentation incomplète ou unité à préciser.,Compléter en priorité les indicateurs qui seront présentés publiquement.
15,Indicateurs marqués validés,OK,0,0 indicateur(s) marqué(s) OUI avec une documentation incomplète.,


Premières variables à documenter ou à relire :


,ID,NIVEAU,COLONNE,LIBELLE,THEME,VALIDE,CHAMPS_A_COMPLETER,LIBELLE_A_RELIRE,DOCUMENTATION_COMPLETE
0,IRIS_10c9faba520ceae9,IRIS,CODGEO_REFERENCE,Codgeo reference,TECHNIQUE,NON,ANNEE ; GEOGRAPHIE ; SOURCE ; DEFINITION,Libellé proche du code technique : relire,False
1,IRIS_70df03d65ef99ef1,IRIS,CODGEO_SOURCE,Codgeo source,TECHNIQUE,NON,ANNEE ; GEOGRAPHIE ; SOURCE ; DEFINITION,Libellé proche du code technique : relire,False
2,IRIS_f6b9a4d49e32f630,IRIS,CODGEO_ETUDE,Codgeo etude,TECHNIQUE,NON,ANNEE ; GEOGRAPHIE ; SOURCE ; DEFINITION,Libellé proche du code technique : relire,False
3,IRIS_659713a84a3448fd,IRIS,DEPARTEMENT,Departement,AUTRES,NON,ANNEE ; GEOGRAPHIE ; SOURCE ; DEFINITION ; UNITE,Libellé proche du code technique : relire,False
4,IRIS_c2c4618895c5989a,IRIS,NOM_IRIS,Nom iris,TECHNIQUE,NON,ANNEE ; GEOGRAPHIE ; SOURCE ; DEFINITION,Libellé proche du code technique : relire,False
5,IRIS_1c678f6cb48fb47a,IRIS,NOM_COMMUNE_SOURCE,Nom commune source,TECHNIQUE,NON,ANNEE ; GEOGRAPHIE ; SOURCE ; DEFINITION,Libellé proche du code technique : relire,False
6,IRIS_606ce6f602f9f6d3,IRIS,TYP_IRIS,Typ iris,AUTRES,NON,ANNEE ; GEOGRAPHIE ; SOURCE ; DEFINITION,Libellé proche du code technique : relire,False
7,IRIS_5753dbde0b3e5bd2,IRIS,NOM_COMMUNE_REFERENCE,Nom commune reference,AUTRES,NON,ANNEE ; GEOGRAPHIE ; SOURCE ; DEFINITION,Libellé proche du code technique : relire,False
8,IRIS_509cea00635bcd7d,IRIS,ORIGINE_SELECTION_J12,Origine selection,TECHNIQUE,NON,ANNEE ; GEOGRAPHIE ; SOURCE ; DEFINITION,,False
9,IRIS_79f7d81e78eba12d,IRIS,NOM_COMMUNE_ETUDE,Nom commune etude,AUTRES,NON,ANNEE ; GEOGRAPHIE ; SOURCE ; DEFINITION,Libellé proche du code technique : relire,False


## 6 — Comprendre les différences entre géographies sources

**Ce que l’on regarde :** certains fichiers utilisent-ils des périmètres communaux différents ?

**Exemple :** une source avant fusion et une source après fusion peuvent partager un code communal
sans décrire la même population.

Une incompatibilité déjà gérée par une valeur indisponible n’est pas, à elle seule, une erreur.
Il faut expliquer cette absence et conserver son statut.

In [9]:
display(audit.verifier_geographies_sources())
if not audit.compatibilite.empty:
    display(audit.compatibilite.loc[
        audit.compatibilite["STATUT"].isin([
            "PERIMETRE_INCOMPATIBLE_MEME_CODE", "CODE_ABSENT_SOURCE"])
    ].head(20))

,Contrôle,Résultat,Nombre concerné,Ce que cela signifie,Action à faire
16,Sources de géographies différentes,À vérifier,2,2 ligne(s) signalent un code absent ou un périmètre incompatible. Ces cas peuvent être correctement gérés ...,Consulter compatibilite_geographies_j16.csv ; ne pas remplacer les données indisponibles par zéro.


,SOURCE,FICHIER,CODGEO,STATUT,MOTIF,REFERENCE
2290,J12,profil_communes_idf_j12.csv,93059,CODE_ABSENT_SOURCE,,https://www.insee.fr/fr/metadonnees/geographie/commune/93066-saint-denis
2295,J12,profil_communes_idf_j12.csv,93066,PERIMETRE_INCOMPATIBLE_MEME_CODE,Fusion Saint-Denis/Pierrefitte : données non réparties entre anciens contours,https://www.insee.fr/fr/metadonnees/geographie/commune/93066-saint-denis


## 7 — Examiner les valeurs et leur disponibilité

Les fichiers thématiques sont lus **un par un** pour limiter la mémoire utilisée.

**Ce que l’on regarde :** les valeurs numériques sont-elles lisibles ? Les règles de bornes
connues sont-elles respectées ? La couverture réelle correspond-elle au catalogue ?

- Une part de population doit rester entre 0 et 100 %.
- Un nombre d’équipements ou une distance doit être positif ou nul.
- Une évolution peut être négative ou dépasser 100 %.
- Un écart, un solde ou un indice demande ses propres règles.
- Une variable sans donnée peut être légitime : le contrôle la signale.

Les effectifs issus du recensement peuvent comporter des décimales : le contrôle ne les arrondit pas.
Le fichier de couverture indique la règle réellement appliquée à chaque variable.

In [10]:
resultats_indicateurs = audit.verifier_valeurs_detaillees()
print("Contrôles des indicateurs :", len(resultats_indicateurs))
display(resultats_indicateurs.loc[resultats_indicateurs["Résultat"].ne("OK")].head(30))
couverture = pd.DataFrame(audit.couverture)
if not couverture.empty:
    display(couverture[["NIVEAU", "THEME", "LIBELLE", "NB_PRESENTES",
                        "NB_TERRITOIRES", "COUVERTURE_PCT"]].sort_values("COUVERTURE_PCT").head(15))

Contrôles des indicateurs : 103


,Contrôle,Résultat,Nombre concerné,Ce que cela signifie,Action à faire
29,COMMUNE — EMPLOI : valeurs numériques,À corriger,4,93 variables examinées ; 4 valeur(s) à corriger selon les règles appliquées.,Lire anomalies_valeurs_j16.csv et vérifier le calcul source.
44,COMMUNE — MENAGES : valeurs numériques,À corriger,9,85 variables examinées ; 9 valeur(s) à corriger selon les règles appliquées.,Lire anomalies_valeurs_j16.csv et vérifier le calcul source.


,NIVEAU,THEME,LIBELLE,NB_PRESENTES,NB_TERRITOIRES,COUVERTURE_PCT
1265,IRIS,TECHNIQUE,Origine selection,168,5264,3.19
952,COMMUNE,RESTAURATION,Part restaurants universitaires dans groupe (%),57,1267,4.50
697,COMMUNE,LOGEMENTS,Part residences universitaires dans groupe (%),57,1267,4.50
193,COMMUNE,AUTRES,Part instituts universitaires dans groupe (%),68,1267,5.37
194,COMMUNE,AUTRES,Part ecoles ingenieurs dans groupe (%),68,1267,5.37
195,COMMUNE,AUTRES,Part enseignement superieur general prive dans groupe (%),68,1267,5.37
196,COMMUNE,AUTRES,Part enseignement superieur agricole dans groupe (%),68,1267,5.37
197,COMMUNE,AUTRES,Part autres structures enseignement superieur dans groupe (%),68,1267,5.37
653,COMMUNE,EQUIPEMENTS,Part ufr composantes dans groupe (%),68,1267,5.37
190,COMMUNE,AUTRES,Part formations sts cpge dans groupe (%),137,1267,10.81


## 8 — Refaire des calculs faciles à expliquer

La densité contrôlée ici est celle du socle J15a :

**Nombre de restaurants ÷ population × 10 000.**

Exemple pédagogique : 12 restaurants pour 5 000 habitants donnent
`12 / 5 000 × 10 000 = 24 restaurants pour 10 000 habitants`.

Le contrôle compare le résultat recalculé à la valeur publiée. Une tolérance de 0,051
permet une densité arrondie à une décimale.
Il vérifie aussi que le nombre de restaurants rapides ne dépasse pas le total.
Les autres ratios du catalogue demandent une vérification de leur définition et de leur dénominateur.

In [11]:
display(audit.verifier_calculs_simples())
if audit.calculs:
    display(pd.DataFrame(audit.calculs).head(20))

,Contrôle,Résultat,Nombre concerné,Ce que cela signifie,Action à faire
143,COMMUNE : densité de restaurants,OK,0,1267 calculs possibles ; 0 valeur(s) incohérente(s) ; 0 densité(s) absente(s).,
144,COMMUNE : restauration rapide et total,OK,0,0 territoire(s) où la restauration rapide dépasse le total.,
145,IRIS : densité de restaurants,OK,0,5208 calculs possibles ; 0 valeur(s) incohérente(s) ; 0 densité(s) absente(s).,
146,IRIS : restauration rapide et total,OK,0,0 territoire(s) où la restauration rapide dépasse le total.,
147,Lecture et bornes des indicateurs du socle,OK,0,0 valeur(s) hors des règles de lecture ou de bornes connues.,


## 9 — Contrôler les données nécessaires aux cartes

**Ce que l’on regarde :** identifiants des arrêts et lignes, relations arrêt–ligne,
coordonnées, couleurs, contours et références des tracés.

Un arrêt associé à plusieurs lignes produit plusieurs relations. Cela n’implique pas plusieurs arrêts physiques.

La structure des GeoJSON est contrôlée. La validité topologique complète et le rendu visuel
restent à vérifier dans les outils de préparation et dans l’application.

In [12]:
display(audit.verifier_cartographie())

,Contrôle,Résultat,Nombre concerné,Ce que cela signifie,Action à faire
148,Arrêts : identifiants,OK,0,0 identifiant(s) vide(s) ou répété(s).,
149,Lignes : identifiants,OK,0,0 identifiant(s) vide(s) ou répété(s).,
150,Relations arrêt–ligne,OK,0,0 relation(s) répétée(s) ou sans arrêt/ligne correspondant(e).,
151,Modes de transport,OK,0,"Modes présents : BUS, METRO, RER, TRAM, TRANSILIEN. 0 valeur(s) non reconnue(s).",
152,Couleurs des lignes,OK,0,0 couleur(s) au format non reconnu.,
153,arrets : coordonnées,OK,0,0 paire(s) de coordonnées invalide(s).,
154,restaurants : coordonnées,OK,0,0 paire(s) de coordonnées invalide(s).,
155,communes_idf.geojson : contours et codes,OK,0,1267 contours ; 0 différence(s) de codes ou de structure.,
156,iris_idf.geojson : contours et codes,OK,0,5264 contours ; 0 différence(s) de codes ou de structure.,
157,Contours IRIS : communes de contexte,OK,0,0 rattachement(s) différent(s) du profil.,


## 10 — Vérifier le registre des sources

**Ce que l’on regarde :** fichier, producteur, année des données, géographie,
date de téléchargement et URL.

Compléter les informations à partir de la source réellement utilisée. La date de téléchargement
et le millésime des données sont deux informations différentes.

In [13]:
display(audit.verifier_sources())

,Contrôle,Résultat,Nombre concerné,Ce que cela signifie,Action à faire
16,Sources de géographies différentes,À vérifier,2,2 ligne(s) signalent un code absent ou un périmètre incompatible. Ces cas peuvent être correctement gérés ...,Consulter compatibilite_geographies_j16.csv ; ne pas remplacer les données indisponibles par zéro.
161,Informations du registre,À vérifier,7,7 ligne(s) avec au moins un champ non renseigné.,Compléter les dates et millésimes réellement connus ; écrire explicitement « non précisé » si nécessaire.


## 11 — Lire un exemple sur Cergy et Pontoise

Ce tableau permet de relire quelques résultats avant de préparer l’étude locale.
Les différences entre nombres et densités doivent pouvoir être expliquées en une phrase.

Une densité renseigne l’offre rapportée à la population résidente. Son interprétation commerciale
demande ensuite les emplois, les déplacements, les équipements et les observations de terrain.

In [14]:
display(audit.apercu_focus())

,Code commune,Commune,Population (habitants),Restaurants recensés,Restauration rapide recensée,Restaurants pour 10 000 habitants
1120,95127,Cergy,69577.99999999996,208.0,138.0,29.894506884360016
1221,95500,Pontoise,31623.0,128.0,75.0,40.476868102330585


## 12 — Vérifier la présence des fichiers de l’application

Le contrôle recherche notamment le composant cartographique et le fichier JavaScript Plotly.
La présence d’un fichier ne prouve pas son bon fonctionnement dans le navigateur :
le bloc suivant prépare les vérifications à faire à l’écran.

In [15]:
display(audit.verifier_fichiers_application())

,Contrôle,Résultat,Nombre concerné,Ce que cela signifie,Action à faire
162,Fichiers nécessaires à la carte,OK,0,Les fichiers contrôlés existent et le JavaScript ne ressemble pas à un pointeur LFS ou à une page HTML.,Tester la carte dans le navigateur : la présence des fichiers ne vérifie pas leur exécution.


## 13 — Faire les tests manuels de Streamlit

Le notebook crée `data/interim/j16/controles_manuels_application_j16.csv` la première fois.
Les exécutions suivantes conservent tes réponses.

Ouvrir l’application et réaliser chaque test. Dans ce CSV, renseigner :

- `OK` si le test a été fait et fonctionne ;
- `A_CORRIGER` si une erreur est constatée ;
- `A_FAIRE` tant que le test n’a pas été réalisé ;
- `NON_APPLICABLE` avec une explication si le test ne concerne pas ta version.

Ajouter une observation courte : par exemple « Cergy et Pontoise affichées ; carte et infobulles lisibles ».
Relancer ensuite le notebook pour inclure les réponses dans le bilan.
Les tests techniques sur les exports ne remplacent pas ces vérifications.

In [16]:
display(audit.preparer_tests_manuels())
print("Fichier à compléter :", audit.interim/"controles_manuels_application_j16.csv")

,TEST,ETAT,OBSERVATION
0,Ouvrir l’application déployée dans une nouvelle session,A_FAIRE,
1,"Afficher Cergy, Pontoise et une autre commune",A_FAIRE,
2,Cliquer sur un IRIS et retrouver le bon territoire,A_FAIRE,
3,Lire les libellés et unités des indicateurs présentés,A_FAIRE,
4,Afficher les transports et vérifier la présence du tramway,A_FAIRE,
5,Vérifier les légendes et les infobulles de la carte,A_FAIRE,
6,Télécharger puis ouvrir un PDF,A_FAIRE,
7,Vérifier la lisibilité sur une petite fenêtre ou un téléphone,A_FAIRE,


Fichier à compléter : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\interim\j16\controles_manuels_application_j16.csv


## 14 — Générer le compte rendu en français

Le texte est généré à partir des résultats réellement obtenus. Il reste prudent lorsque
des points sont à corriger ou des tests sont non réalisés.

Tu peux ajouter un commentaire personnel dans `MON_COMMENTAIRE`. Pour rédiger facilement,
réponds à ces trois questions :

1. Quels indicateurs sont suffisamment documentés pour être présentés ?
2. Quels points dois-je encore corriger ou expliquer ?
3. Quelle sera ma priorité au J17 ?

La synthèse est enregistrée dans `outputs/rapports/j16/synthese_j16.txt`.
Le fichier `bilan_j16.html` présente les mêmes résultats dans un navigateur.

In [17]:
MON_COMMENTAIRE = ""
# Exemple de structure à adapter après lecture des résultats :
# "Je retiens pour ma démonstration ... Les points à traiter sont ... Ma priorité au J17 sera ..."

bilan, texte_synthese = audit.exporter(commentaire=MON_COMMENTAIRE)
print(texte_synthese)
print("Rapport HTML :", audit.rapports/"bilan_j16.html")
print("Compte rendu texte :", audit.rapports/"synthese_j16.txt")
print("État du bilan :", bilan["statut"])

Bilan J16 — Contrôle des données et de l’application

Le contrôle porte sur 1267 communes, 5264 IRIS et 1348 variables du catalogue.
Les vérifications concernent les fichiers, les codes géographiques, la documentation, la lecture des valeurs, quelques calculs simples et la structure des cartes.

Résultats : 157 contrôles OK ; 4 à vérifier ; 2 à corriger ; 9 non testés.
Parmi les non testés, 8 demandent encore une action dans ce bilan. La topologie complète et les tests déclarés non applicables restent hors du périmètre exécuté.
État du bilan : A_CORRIGER.

Les points « À corriger » doivent être traités avant de présenter les résultats concernés.
Les points « À vérifier » demandent une relecture ou une documentation supplémentaire. Les tests manuels de l’application restent à réaliser s’ils sont marqués A_FAIRE.

Ce contrôle technique ne valide pas automatiquement les définitions statistiques. Les valeurs manquantes restent manquantes. Les médianes et pourcentages ne sont pas additionné

## 15 — Traiter les actions prioritaires

Commencer par les lignes **À corriger**, puis documenter les indicateurs qui seront utilisés
dans l’étude locale. Un résultat technique OK ne modifie jamais le champ `VALIDE` du catalogue.

Le fichier `documentation_indicateurs_j16.csv` peut contenir de nombreuses lignes à compléter.
Le travail peut avancer par thématique, en commençant par les indicateurs réellement présentés.

In [18]:
a_traiter = audit.afficher()
display(a_traiter.loc[a_traiter["Résultat"].isin(["À corriger", "À vérifier"])].head(30))
print("Le détail complet est disponible dans bilan_j16.html et bilan_controles_j16.csv.")

,Contrôle,Résultat,Nombre concerné,Ce que cela signifie,Action à faire
13,Libellés compréhensibles,À vérifier,104,104 libellé(s) à relire selon les règles de présentation.,Consulter documentation_indicateurs_j16.csv ; appliquer le correctif J15b puis relire les libellés concernés.
14,"Sources, années, définitions et unités",À vérifier,1348,1348 variable(s) avec documentation incomplète ou unité à préciser.,Compléter en priorité les indicateurs qui seront présentés publiquement.
16,Sources de géographies différentes,À vérifier,2,2 ligne(s) signalent un code absent ou un périmètre incompatible. Ces cas peuvent être correctement gérés ...,Consulter compatibilite_geographies_j16.csv ; ne pas remplacer les données indisponibles par zéro.
29,COMMUNE — EMPLOI : valeurs numériques,À corriger,4,93 variables examinées ; 4 valeur(s) à corriger selon les règles appliquées.,Lire anomalies_valeurs_j16.csv et vérifier le calcul source.
44,COMMUNE — MENAGES : valeurs numériques,À corriger,9,85 variables examinées ; 9 valeur(s) à corriger selon les règles appliquées.,Lire anomalies_valeurs_j16.csv et vérifier le calcul source.
161,Informations du registre,À vérifier,7,7 ligne(s) avec au moins un champ non renseigné.,Compléter les dates et millésimes réellement connus ; écrire explicitement « non précisé » si nécessaire.


Le détail complet est disponible dans bilan_j16.html et bilan_controles_j16.csv.


## 16 — Ajouter J16 à GitHub

Ajouter les fichiers de travail suivants :

- `notebooks/16_controler_donnees_application.ipynb` ;
- `scripts/controle_j16.py` ;
- `docs/README_J16.md`.

Le compte rendu texte et le rapport HTML peuvent être ajoutés après relecture,
en vérifiant qu’ils correspondent au lot d’exports réellement utilisé.
Les sorties volumineuses du notebook peuvent être effacées avant son ajout au dépôt.

Commit proposé :

`feat: ajouter les contrôles J16 et un bilan qualité compréhensible`

Les exports J15a/J15b ne sont pas à remplacer simplement parce que J16 a été exécuté.
Si tu corriges une source ou un catalogue après ce bilan, régénère les exports concernés,
applique les correctifs de libellés et relance J16.

**Étape suivante : J17 — enrichir les commerces, cinémas et pôles d’attractivité.**
Le premier partage de l’étude élargie reste prévu vers J24, après les transports,
la fréquentation, les zones accessibles et l’intégration dans l’application.